# Broken-Tether — a quantitative teardown 🔬
### Spread half-life · the causal z-score book · in-sample vs out-of-sample · the spurious-pair false-positive rate · hedge-ratio drift

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Stays tethered?: Breaks](https://img.shields.io/badge/Stays_tethered%3F-Breaks-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §3.8 pairs trading: trade the mean reversion of a cointegrated spread. We prove the engine on a synthetic cointegrated pair (and a spurious one), then show real ETF pairs break out of sample.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from broken_tether import data, spread, strategy, decompose, extension

# Offline synthetic pairs: a COINTEGRATED pair (stationary spread, tradable) and a SPURIOUS one (two
# independent random walks that drift together). The real ETF verdict is in ../docs/results.md.
coint, _    = data.synthetic_pair(revert_rho=0.93, seed=23)   # genuinely cointegrated
spur,  _    = data.synthetic_pair(revert_rho=1.0,  seed=23)   # spurious (null)
print("two synthetic pairs: one cointegrated, one spurious")


two synthetic pairs: one cointegrated, one spurious


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — is the spread tradably stationary? | 🟡 `WEAK` | The half-life cleanly flags a baked cointegrated pair, but real ETF pairs are weakly/unstably cointegrated and the trade is thin even at best. |
| **Tradability** | 🔴 `MIRAGE` | Of **45** real pairs, **1** survives out of sample; first-/second-half rank correlation **+0.17**. |
| **Stays tethered?** | ⚪ `Breaks` | The best in-sample pair (QQQ/EWJ) drops **+0.61 → +0.18**; the hedge ratio drifts **209%** of its level; **{R['spur_fp']}%** of random walks look cointegrated by chance. |

> **In one sentence:** a real convergence trade on a *stable* cointegrated pair, but stable cointegration among liquid ETFs is rare, drifting, and mostly an artefact of selection — so the scanned-universe edge breaks out of sample.

*(This notebook executes on synthetic pairs; the real ETF numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Two log-prices are **cointegrated** if $\log A_t - \beta \log B_t = s_t$ is stationary. Stationarity ⇔ a finite mean-reversion **half-life** $-\ln 2/\ln(1+\rho)$ from the AR(1) $\Delta s_t = a + \rho s_{t-1} + e$ ($\rho<0$). The trade is $w_t = -\,\mathbb{1}[z_t>\text{entry}] + \mathbb{1}[z_t<-\text{entry}]$ on the spread, with a **causal** trailing β and z. The synthetic spread is AR(1) with persistence `revert_rho`; `=1` is the spurious (random-walk-spread) null.

In [2]:
for label, px in [('cointegrated', coint), ('spurious', spur)]:
    st = spread.stationarity(px['A'], px['B'])
    print(f"{label:12s}: hedge ratio {st['hedge_ratio']:.2f}, half-life {st['half_life_days']:.0f}d, reverting={st['is_reverting']}")

cointegrated: hedge ratio 0.70, half-life 11d, reverting=True


spurious    : hedge ratio 0.45, half-life 82d, reverting=True


## Beat 2 · So what?

Cointegration is special because the spread is mean-reverting *unconditionally* — market-neutral by construction. But it is estimated, and three things can void it: the relationship **drifts** (β non-constant), the apparent cointegration is **selection** (found by scanning), or it is real but **competed away**. Beats 4–6 test all three; the half-life is the one clean in-sample signal, the out-of-sample split the decisive one.

## Beat 3 · Pre-registered protocol

1. **Half-life** (`spread.half_life`): short on the cointegrated tape, near-∞ on spurious.
2. **In/out-of-sample** (`decompose.in_sample_vs_oos`): first- vs second-half Sharpe.
3. **Selection** (`decompose.spurious_pairs`): false-positive rate among independent walks.
4. **Drift** (`extension.hedge_ratio_drift`): trailing β dispersion.

**Breaks line:** OOS Sharpe collapses, in-sample rank doesn't carry, β drifts, FP rate non-trivial.

## Beat 4 · The teardown

### 4a · Half-life and the trade, cointegrated vs spurious

In [3]:
for label, px in [('cointegrated', coint), ('spurious', spur)]:
    oos = decompose.in_sample_vs_oos(px['A'], px['B'], cost_bps=2.0)
    cmp = strategy.compare(px['A'], px['B'], cost_bps=2.0)
    print(f"{label:12s}: Sharpe {cmp['sharpe']:+.2f} ({cmp['trades']} trades) | "
          f"IS {oos['first_half_sharpe']:+.2f} -> OOS {oos['second_half_sharpe']:+.2f} (survives={oos['survives_oos']})")

cointegrated: Sharpe +0.29 (96 trades) | IS +0.50 -> OOS +0.13 (survives=False)
spurious    : Sharpe +0.22 (75 trades) | IS +0.25 -> OOS +0.11 (survives=False)


### 4b · The spurious-pair false-positive rate

In [4]:
for hl in [40, 60, 90]:
    sp = decompose.spurious_pairs(n_series=25, hl_threshold=hl, seed=1)
    print(f"threshold half-life < {hl}d: {sp['false_positive_rate']:.0%} of independent walks 'look cointegrated' ({sp['n_look_cointegrated']}/{sp['n_pairs']})")

threshold half-life < 40d: 0% of independent walks 'look cointegrated' (1/300)


threshold half-life < 60d: 4% of independent walks 'look cointegrated' (11/300)


threshold half-life < 90d: 13% of independent walks 'look cointegrated' (38/300)


> 💡 **In plain words.** Even with *no* relationship, a fraction of random pairs will show a tradable-looking half-life by chance — and the looser your screen, the more you let through. Scan thousands of pairs and your 'best' ones are mostly these flukes.

### 4c · Out-of-sample on real ETFs (quoted)

Across **45** real ETF pairs only **1** survives the second half; the best in-sample pair **QQQ/EWJ** drops **+0.61 → +0.18**, and the first-/second-half Sharpe rank correlation is **+0.17** — see [`../docs/results.md`](../docs/results.md).

## Beat 5 · The verdict

- **Engine works** (4a): half-life flags real cointegration; spurious collapses OOS.
- **Selection is real** (4b): random walks pass a cointegration screen by chance.
- **Real pairs break** (4c): 1/45 survive, β drifts 209%.

> **Signal `WEAK` · Tradability `MIRAGE` · Stays tethered? `Breaks`.**

## Beat 6 · Could you trade it?

- **Scanned pairs are cherry-picked flukes** that revert to nothing live.
- **β drifts**, so the spread anchor slides under the trade.
- **A real pair pays thinly** and the classic edge has decayed (Gatev et al. → Do & Faff).

Tradability **`MIRAGE`** on a scanned universe.

## Beat 7 · Going further

### 7a · Worked complement — the tether drifts, winners don't repeat
The trailing β dispersion and the in/out-of-sample rank correlation across real pairs.

In [5]:
for label, px in [('cointegrated', coint), ('spurious', spur)]:
    d = extension.hedge_ratio_drift(px['A'], px['B'])
    print(f"{label:12s}: trailing beta {d['beta_min']:.2f}..{d['beta_max']:.2f} (drift {d['beta_rel_drift']:.0%} of mean)")
print('On real ETF pairs (../docs/extension.md): median beta drift 209%, IS/OOS rank corr +0.17 --')
print('the in-sample winners are not the out-of-sample ones.')

cointegrated: trailing beta -1.12..0.54 (drift 79% of mean)
spurious    : trailing beta 0.11..1.72 (drift 33% of mean)
On real ETF pairs (../docs/extension.md): median beta drift 209%, IS/OOS rank corr +0.17 --
the in-sample winners are not the out-of-sample ones.


**The result.** The hedge ratio that defines each pair wanders, so the spread reverts toward a moving anchor; and the first-half ranking barely predicts the second (rank corr +0.17). Selecting pairs on in-sample performance harvests noise — only 1/45 survive. The one assumption the trade can't do without — a stable relationship — is exactly the one liquid markets won't grant. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **Structurally-linked pairs** (dual share classes, ADR vs local line, closed-end fund vs NAV) have an economic anchor — test whether those hold where scanned ETF pairs don't.
- **Engle–Granger / Johansen** tests with a multiple-testing correction to size selection.
- **Kalman-filtered dynamic hedge ratio** — does adapting β to the drift rescue anything, or just chase noise?

PRs welcome — find a structurally tethered pair that survives, or quantify the decay.